In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from functools import partial

from folktexts.acs import ACSDataset, ACSTaskMetadata
from folktexts.llm_utils import load_model_tokenizer
from folktexts.classifier import TransformersLLMClassifier
from folktexts.prompting import encode_row_prompt_few_shot


In [3]:
task_name = 'ACSIncome'
task = ACSTaskMetadata.get_task(task_name)
acs_dataset = ACSDataset.make_from_task(
    task=task,
    cache_dir='./data',)

In [4]:
X_sample, _y_sample = acs_dataset.sample_n_train_examples(n=1)

In [9]:
model_name_or_path = "../models/meta-llama--Meta-Llama-3-8B-Instruct"

model, tokenizer = load_model_tokenizer(model_name_or_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [84]:
n_shots = 4
llm_clf = TransformersLLMClassifier(
    model=model,
    tokenizer=tokenizer,
    task=task,
    batch_size=50,
    context_size=750,
    encode_row=partial(
        encode_row_prompt_few_shot,
        task=task,
        dataset=acs_dataset,
        n_shots=n_shots,
        class_balancing=True,
        reuse_examples=True,
        prompt_variation={"format": "bullet", "connector": "is"},
    ),
)


print(llm_clf.encode_row(X_sample.iloc[0], question=llm_clf.task.question))

class_balancing: True
ys: [0 0 1 1]
i A
i A
i B
i B
-------- 0 ------Information:
- age is 51 years old
- class of worker is Working for a for-profit private company or organization
- highest educational attainment is 11th grade
- marital status is Married
- occupation is Miscellaneous Production Workers, Including Equipment Operators And Tenders
- place of birth is Mexico
- relationship to the reference person in the survey is The reference person itself
- usual number of hours worked per week is 20 hours
- sex is Male
- race is White

Question: What is this person's estimated yearly income?
A. Below $50,000.
B. Above $50,000.
Answer:-------- 1 ------Information:
- age is 53 years old
- class of worker is Owner of non-incorporated business, professional practice, or farm
- highest educational attainment is Bachelor's degree
- marital status is Married
- occupation is Musicians and Singers
- place of birth is New York
- relationship to the reference person in the survey is The referenc

In [20]:
for n_shots in [4, 10, 50, 100]:
    llm_clf = TransformersLLMClassifier(
        model=model,
        tokenizer=tokenizer,
        task=task,
        encode_row=partial(
            encode_row_prompt_few_shot,
            task=task,
            dataset=acs_dataset,
            n_shots=n_shots,
            class_balancing=True,
            reuse_examples=True, # make sure these are always the same examples
            prompt_style={"format": "bullet", "connector": "is"},
        ),
    )

    num_reps = 50 
    repeated_sample = pd.concat([X_sample]*num_reps)
    predictions_repeated_sample = llm_clf.predict_proba(repeated_sample)[:,1] #returns multi-class probs
    
    plt.hist(predictions_repeated_sample, weights= np.zeros((num_reps, )) + 1./num_reps);plt.ylim(0,1)
    plt.title(f'{n_shots} shots')
    plt.show()

Computing risk estimates:   0%|          | 0/4 [00:00<?, ?it/s]

TypeError: encode_row_prompt_few_shot() got an unexpected keyword argument 'prompt_style'